### Importa los datos
Dataset con diferentes fármacos, sus efectos y ratings de los clientes.

Importa el dataset *drugLibTrain_raw.tsv*

"\n1. Importación de datos — Carga el .tsv con sep='\t' (3107 filas, 9 columnas).\n2. Selección de columnas manejables\n\nNuméricas: rating\nCategóricas con <10 valores únicos: effectiveness (5 cats) y sideEffects (5 cats)\nSe descartan urlDrugName (502 valores), condition (1426) y las columnas de texto libre\n\n3. Encoding con dummies — Las dos columnas categóricas se convierten en 8 columnas binarias (0/1). El resultado final es una matriz de 9 features.\n4. Escalado con StandardScaler — Imprescindible para K-Means, ya que el algoritmo trabaja con distancias.\n5. Búsqueda de la mejor K con Silhouette Score — Prueba K de 2 a 10. El score más alto resultó ser K=10 (0.66), con una mejora progresiva. Se representan también con el método del codo para confirmarlo.\n6. Modelo final y visualizaciones\n\nPie chart con la distribución de los clusters\nBar chart con el rating medio por cluster\nPerfil de cada cluster: rating medio, efectividad más frecuente y efectos secundarios más frecuentes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')



df = pd.read_csv('data/drugLibTrain_raw.tsv', sep='\t')
# el archivo es .tsv, que son datos separados por tabulaciones, asíq ue pongo sep='\t'
# para que pandas lo entienda bien


print('Shape:', df.shape)
df.head()

Shape: (3107, 9)


,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,2202,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,3117,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,1146,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,3947,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,1951,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above


### Descriptive Analysis

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3107 entries, 0 to 3106
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         3107 non-null   int64 
 1   urlDrugName        3107 non-null   object
 2   rating             3107 non-null   int64 
 3   effectiveness      3107 non-null   object
 4   sideEffects        3107 non-null   object
 5   condition          3106 non-null   object
 6   benefitsReview     3089 non-null   object
 7   sideEffectsReview  3032 non-null   object
 8   commentsReview     3095 non-null   object
dtypes: int64(2), object(7)
memory usage: 218.6+ KB


In [15]:
df.describe()

# muestra las únicas dos columnas numñericas de df

# hay 4162 filas en el archivo original, min 0 máx 4161
# df train tiene 3107 de ellas
# el resto imagino que estaránen un test

# valoración media del paciente es un 7 sobre 10
# notas se dispersan bastante, casi 3 puntos arriba o abajo de la media
# 25% pacientes dio un 5 o menos
# mediana ye 8, 50% pacietnes dio un 8 o más
# 75% pacientes dio un 9 o menos

# la media ye menor que la mediana
# sigfca distribución está sesgada hacia la izquierda, la mayoría de pacientes puntúa alto, de ahí la mediana en 8
# luego minoría que puntúa muy bajo (1-3), y esos valores bajos "tiran" de la media hacia abajo hasta el 7

# real life: la gente que está satisfecha con su fármaco tiende a puntuar alto, pero hay casos donde el medicamento
# no funcionó o tuvo efectos secundarios graves, y esos pacientes puntúan muy bajo

,Unnamed: 0,rating
count,3107.000000,3107.000000
mean,2080.607016,7.006115
std,1187.998828,2.937582
min,0.000000,1.000000
25%,1062.500000,5.000000
50%,2092.000000,8.000000
75%,3092.500000,9.000000
max,4161.000000,10.000000


Quedate únicamente con las columnas que podamos manejar: Columnas numéricas y columnas categóricas con pocas categorías (menos de 10)

In [16]:
# separo cols numéricas, menos el índice 'Unnamed: 0', que no aporta
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if 'Unnamed' not in c]
print(num_cols)

# cols categóricas con menos de 10 categorías únicas
categ_cols = df.select_dtypes(include=['object']).columns.tolist()
categ_ok = [col for col in categ_cols if df[col].nunique() < 10]
print(categ_ok)

for col in categ_ok:
    print(f'{col} ({df[col].nunique()} categorcas): {df[col].unique()}')
    
    
    
# para K-Means descarto lo que no puedo usar:
# reseñas escritas por pacientes
# urlDrugName tiene 502 fármacos distintos y condition tiene 1426 condiciones médicas → demasiadas categorías, también fuera

# me quedo con: rating, effectiveness y sideEffects

['rating']
['effectiveness', 'sideEffects']
effectiveness (5 categorcas): ['Highly Effective' 'Marginally Effective' 'Ineffective'
 'Considerably Effective' 'Moderately Effective']
sideEffects (5 categorcas): ['Mild Side Effects' 'Severe Side Effects' 'No Side Effects'
 'Extremely Severe Side Effects' 'Moderate Side Effects']


In [17]:
# me qeudo sólo con las cols que peudo manejar
cols_si = num_cols + categ_ok
df_kmeans = df[cols_si].dropna().copy() 
# creo nuevo df con solo las 3 columnas que me interesan
# de paso elimino cualquier fila que tenga algún valor vacío con el dropna())
# es como quedarte solo con las páginas de un libro que necesitas y tirar las que están en blanco

print(cols_si)

['rating', 'effectiveness', 'sideEffects']


In [ ]:
print(df_kmeans.shape)

(3107, 3)


In [19]:
df_kmeans.head()

,rating,effectiveness,sideEffects
0,4,Highly Effective,Mild Side Effects
1,1,Highly Effective,Severe Side Effects
2,10,Highly Effective,No Side Effects
3,3,Marginally Effective,Mild Side Effects
4,2,Marginally Effective,Severe Side Effects


#### Transforma las columnas categóricas

Transforma las columnas categoricas a numericas mediante dummies

In [ ]:
df_kmeans['effectiveness'].value_counts()


effectiveness         
Highly Effective          1330
Considerably Effective     928
Moderately Effective       415
Ineffective                247
Marginally Effective       187
Name: count, dtype: int64

In [29]:
df_kmeans['sideEffects'].value_counts()

sideEffects
Mild Side Effects                1019
No Side Effects                   930
Moderate Side Effects             614
Severe Side Effects               369
Extremely Severe Side Effects     175
Name: count, dtype: int64

In [33]:
# pd.get_dummies convierte cada categoría en una columna binaria (0/1)
# drop_first=True elimina una dummy por variable para evitar multicolinealidad
df_encoded = pd.get_dummies(df_kmeans, columns=categ_ok, drop_first=True)

df_encoded.head()

,rating,effectiveness_Highly Effective,effectiveness_Ineffective,effectiveness_Marginally Effective,effectiveness_Moderately Effective,sideEffects_Mild Side Effects,sideEffects_Moderate Side Effects,sideEffects_No Side Effects,sideEffects_Severe Side Effects
0,4,True,False,False,False,True,False,False,False
1,1,True,False,False,False,False,False,False,True
2,10,True,False,False,False,False,False,True,False
3,3,False,False,True,False,True,False,False,False
4,2,False,False,True,False,False,False,False,True


In [ ]:
df_encoded.shape

(3107, 9)

In [32]:
df_encoded.columns.tolist()

['rating',
 'effectiveness_Highly Effective',
 'effectiveness_Ineffective',
 'effectiveness_Marginally Effective',
 'effectiveness_Moderately Effective',
 'sideEffects_Mild Side Effects',
 'sideEffects_Moderate Side Effects',
 'sideEffects_No Side Effects',
 'sideEffects_Severe Side Effects']

In [ ]:
from sklearn.preprocessing import StandardScaler

# como k-means es sensible a la escala de las variables, voy escalar los datos
scaler = StandardScaler()
X = scaler.fit_transform(df_encoded)

X.shape

# ene ste caso tengo 'rating' con valores del 1 al 10 y las dummies con valores de 0 a 1
# k-means calcula distancias entre puntos, y si una variable tiene números mucho más grandes que otra,
# esa variable domina el resultado injustamente, como si en un partido de fútbol un equipo
# jugara con 20 jugadores

# StandardScaler iguala el terreno de juego: transforma todas las variables para que tengan media 0 y desviación estándar 1
# así todas las variables pesan igual


(3107, 9)

#### Evalua cual es la mejor K

Utiliza silhouette_score para evaluar cual es la mejor K.

El **Silhouette Score** mide qué tan bien está clasificado cada punto:
- **Cercano a +1**: el punto está bien dentro de su cluster y lejos de los demás ✅
- **Cercano a 0**: el punto está en la frontera entre clusters ⚠️
- **Cercano a -1**: el punto probablemente está en el cluster equivocado ❌

$$S = \frac{b - a}{\max(a, b)}$$

Donde $a$ = distancia media intra-cluster y $b$ = distancia media al cluster vecino más cercano.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Probamos K de 2 a 10 y almacenamos Silhouette Score e Inercia
k_range = range(2, 11)
silhouette_scores = []
inertias = []
kmeans_per_k = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X)
    score = silhouette_score(X, kmeans.labels_)
    silhouette_scores.append(score)
    inertias.append(kmeans.inertia_)
    kmeans_per_k.append(kmeans)
    print(f'K={k:2d} | Silhouette Score: {score:.4f} | Inertia: {kmeans.inertia_:.2f}')

best_k_idx = silhouette_scores.index(max(silhouette_scores))
best_k = list(k_range)[best_k_idx]
print(f'\n✅ Mejor K según Silhouette Score: K = {best_k} (score = {max(silhouette_scores):.4f})')

In [ ]:
# Visualizamos Silhouette Score y Método del Codo juntos
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Silhouette Score ---
axes[0].plot(list(k_range), silhouette_scores, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=best_k, color='red', linestyle='--', linewidth=1.5, label=f'Mejor K = {best_k}')
axes[0].set_xlabel('K (número de clusters)', fontsize=13)
axes[0].set_ylabel('Silhouette Score', fontsize=13)
axes[0].set_title('Silhouette Score por K\n(más alto = mejor separación de clusters)', fontsize=12)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# --- Elbow Method (Inercia) ---
axes[1].plot(list(k_range), inertias, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('K (número de clusters)', fontsize=13)
axes[1].set_ylabel('Inercia', fontsize=13)
axes[1].set_title('Método del Codo (Elbow Method)\n(buscar el "codo" de la curva)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Genera el K-Means

In [ ]:
# Entrenamos el modelo final con la mejor K encontrada
kmeans_final = KMeans(n_clusters=best_k, n_init=10, random_state=42)
kmeans_final.fit(X)

# Añadimos la etiqueta de cluster al dataframe original
df_clean['cluster'] = kmeans_final.labels_

print(f'Modelo entrenado con K = {best_k}')
print(f'Inercia final: {kmeans_final.inertia_:.2f}')
print(f'Silhouette Score final: {silhouette_score(X, kmeans_final.labels_):.4f}')

Comprueba los resultados y muestra en un pie plot la distribución de los distintos clusters.

In [ ]:
# --- Análisis de los clusters ---
print('=== Distribución de instancias por cluster ===')
cluster_counts = df_clean['cluster'].value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    pct = count / len(df_clean) * 100
    print(f'  Cluster {cluster_id}: {count:4d} muestras ({pct:.1f}%)')

print('\n=== Media de variables numéricas por cluster ===')
print(df_clean.groupby('cluster')[numeric_cols].mean().round(2))

In [ ]:
# --- Análisis categórico por cluster ---
print('=== Moda de effectiveness por cluster ===')
print(df_clean.groupby('cluster')['effectiveness'].agg(lambda x: x.value_counts().index[0]))

print('\n=== Moda de sideEffects por cluster ===')
print(df_clean.groupby('cluster')['sideEffects'].agg(lambda x: x.value_counts().index[0]))

In [ ]:
# --- Pie plot de distribución de clusters ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
labels = [f'Cluster {i}' for i in cluster_counts.index]
colors = plt.cm.Set3(np.linspace(0, 1, best_k))

axes[0].pie(
    cluster_counts.values,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    textprops={'fontsize': 11}
)
axes[0].set_title(f'Distribución de Clusters (K={best_k})', fontsize=13, fontweight='bold')

# Bar chart con media de rating por cluster
mean_rating = df_clean.groupby('cluster')['rating'].mean().sort_index()
bars = axes[1].bar(mean_rating.index, mean_rating.values, color=colors, edgecolor='black', linewidth=0.7)
axes[1].set_xlabel('Cluster', fontsize=12)
axes[1].set_ylabel('Rating medio', fontsize=12)
axes[1].set_title('Rating medio por Cluster', fontsize=13, fontweight='bold')
axes[1].set_xticks(mean_rating.index)
axes[1].set_xticklabels([f'C{i}' for i in mean_rating.index])
axes[1].grid(axis='y', alpha=0.3)

# Anotamos el valor sobre cada barra
for bar, val in zip(bars, mean_rating.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()